# GwenLand glcuda Wave 121 - LM-head tensor-core GEMM gate

Six counterbalanced T4 event-timed pairs with the production oracle.


In [ ]:
import base64
import gzip
import hashlib
import json
import math
import os
from pathlib import Path
import random
import re
import shutil
import statistics
import subprocess
import traceback
import urllib.request
import zipfile

BUILD = "wave121-in-process-stability-v1"
REPO_URL = "https://github.com/gwenland-org/gwenland-ai.git"
BASE_REV = "5de5be39c0190b9367da18f0f2001e7f40208c92"
SOURCE_REV = "f433840605b03142a653f2ed15d6d69f331b37db"
PATCH_SHA256 = "a224893b8492b3592126378b47700d22ff13f124daaabdd30a3c59ebcd97f855"
PATCH_GZIP_B64 = """H4sIAPsGpmoC/8086XrcOHL/9RSwNtGyV2yq70sr78q2xjsZXyNpZpJ4/FFsEpS44tHmoWNtfV8eIk+YJ0lVASDBJrulcSb5oh/qbgIoAFWFukEv8H3W7V4GOXMOLkO38JwDfudEq5BnB7fODe/3Z3YQ26s0cXmW2VnuLIMwyO+tNGPL3zpiJ+a3zA9CzqLE46zf601Go50g9vgd6z3xz7KWwyH3hkN35Hjj5XDg9t05n06mw74zHrkjd96b+cOl5092ut0uO/D4zUFchOHO/v7+N6z4r39l3Z7ZY/t9sz8esb/+dWf/4OAZ+wWGMRjHgrgrxzH49Ao3D5KYlSDYlZPG0GjRMDH2fSz2H5osLeKYpyZ7+dOrYxgfRE56z9wkzvldbjIn9pgThonrEFAnzHkaOzln+RUXoFKeO0HMPerqcZ+nKfe6Kc8Cr3BCtnLyq8xiL5MogvHa+vzQucyYk3KWFatVGHBPwFveI2zmwqw8ZUvuJ9BF7Q82leaHskORAfzsNsjdKxZkjN8BFBe46IqnHDa7s19knAGyAcBiweNLWKWdp06QLxZfXocn9MBk38ew5O/jVZE/HJZDgD7UCb+8TGI/uDSZ+CWGVV1xZYvFa/oUbYc4NSAwy9mH0/dvP5zbP737/nzB9rI8ZUds9y13siJFDMKiPQ4IjYI4yPLAZdl9lvOIyBitcthiyn3gm3uLncDmAM3sKrlleXLNgeROiigKgfw5vwRURU6eBncsKsI8QEwIisEqAW3AAkChiEdJem+ycx5nSQo0ASoJEvvBHbSHThEDLi95EvE8hVl3D9VOfvzp+PT85PxsAQCDf3DYx7hXNv5yfPr2pw9n9oeTUxu+an3KLif/+uHk5fnJK1ui5Pz9DyfvNGiD0Yjw5sewH6CFEXgZoOxjMRx86rDuc41M7MvOPoO/5hP8I+TYNBr+WXli33DX6JhVj8i5s0EI2NQTuvW1NsD+iqdODvRZsJ7V05uSlX29YKP1ZyvsOB9rT1O+4k5ur3gMx+UeJrD0KSyrWvhiAQfGAYIZHdHhYWf/QaIBzqXtpJEhGgT7AkZ0LpRQNVTJJy7QNPDgmC7YMklC+TRJHTeER9ARnhBWT3kGs//Zn4xM9iK5+7N3j4LDg+OSpkm6WJzgx/PnCsFiFVbGc3vJgVVAVlzbdOZt349tdeiNcv7OXw7FyJDnLAFKHSkYASLBKGndKXsGPna0xBGQRGLPjjZwEPv6lbqXZLcckJ7A/dzo4KiPYtOfdBYBiVWkMYO9GSBc4Ng8M6pG/NsVg1gUZNDqXi1ATEVHXx5YbVH4QH1b/OUB5Q93c+4dffzy8GnXrIOEXZVIYV/Ybvljl8HIMKOHSpTCs7XhDYy0tJcoWG+jzWgPOxozgvBIjE7nsGQ//Hh/bYgJOSjK0I6yTsWWEazQ0HkHztZTWAcZIAIGcFKQ+keyZ3yzWOADo2Nl18HK6Hc0diH9BF2xg7biGPSSoW8hubaT1NgFbXAJzL1NmbK371+dvGEfna/LT7s11iRbQEwlJ9AaU37D0wzbiRt49szA/shpHqgaHzFwBiLT2F3udvSBsAC0NOwkdp8yWnYXMGpQUBkc6drEEjLGmNUQBui9XBXQUxcSiwVoyCvbJS1m6CpNPxMZ595CrGM00KXltnMuB/gO8K8a86BWBCsB7goAlX/RnoSJ49lEWWOPPmpkAOaF1WM/mBVabfcKNrknMKBLCDzoIY/FGd8gGX7TiVcaF+weOC4589LAzxcMjjhM8OWhcZzV/GvP25fyxKNXp6QrKVkZFosFqK2SM90Gft1H8StEgS0EsQBQSeG9mhiuukPXP1UDtQ2UAkd75gdp1no8haWkQF7yGNUs2BxxIoTobm0b2VWRe8ltbFRHAQiniVAwhT4S35ksT4u6fMeeNvYAQ33NNNG7EWNIJbsHPGcyxIBZTWLK1ZYr00RkSa6DA/Z+mfH0hsytbhKH96DiwHpiqwQoTIuRlvrcYj9wvmIJGN9wfuEUQUcYdsNLUPoGgTIcBQdsBO3d0gROwGIDZUOGeKhkDOEGTnEIKqiEFuQ1j+AaRoCqYSgPkAVgDaFzH8SXBH/Q63Uz8kg01wHwY5WHribONEQKIaV0BbBLG1aRSq0IxdE5rJtMTnH66zSyyladr6gFsGXD2mPja/6VlQprvZfkQEGFQU+tVZu1iJ0bJwidJUpffXHgD8V5GDfExUdSM4NeV+IE+O/Lr7s1Df3r7uLLg/nr7lWSgS1YogcfL6w5tsBOH2/IV7Jlgi3yHNIMNMHDk0VTNdNaQ4kGONC509KhErZOxsBUZH9CtxmsY3bw+OCm9XFYP6nAbJeCyUtQ9ChbP6sbaFGnBw0V1IidiAOOft398vDrLuBOLVBDNDqZJaGW9znPQLc5nnoSOW62AcukM3EuC6fZ2LgBKVUHWsHG1mpJVhHfps4KGbnX2dgfF7ytZ6cpyJRirkncNb0J9uAGRRWjNkUjTRfG6j89+tQ0pDzsr3UpP0iS1/QPWEjQGWSPssG+VEBKu1ku4kE3JOIbFbJYGw42mmZxO7vlMOTFz4WT5jQcdYfyedd1i7FKsgBha7qig2NovaTX7QDEM1i1PC4iUnRweNbY+UlSc7suUmBglNjmb3IxDn/r8erPpIIQ56vCsX7KJArVEVKoUr9hqXrvVonZKhO/WR6SBCuX2tIoV9zSUtK52QT7aJtng+zdKn8fE7FbBzbE64ZjLj9aiFwRF46IF7h5l8zBJ9C4JJjgisxegX8gCCzp2EJcsWAJ7QAB1knWSip1ElseAaYG+vAWCmhdR9sf675LQyAKKYiesNcSsc5S90AaWAdgeNdi0802GXBeDkb+fDl33El/Mpz3lt5y1oMf/nA4WE77wyWfeXN35o8tazKbjea+6/VnU8cdzfzheMA9153w6RiejIbzcY+P+j1HBbQx7rxlbfVIdEs7xpwn5oTtT8wpRpwZhmbJ8AMioxS8dVKPrRwwSA0QifgQVFXesXbYDsYchYPv+8Fi4do3SeDJgCk9zu5jFxz/PIkC+PxyTF9eYKCKvUcRCiYpSCcJSAVwX4cUUcDnuLjBYAwr2x8MxALZqlgC8BRMXTCycStnHOOCQr/xKLLj/oSMKk4BHREWY8JQPlAWen8hItiMO+4V6JW46wdoZH/33TtWRrRRvpPFjIECBhY0T/+YVaCcEPX1fdcvUEk5eQ5GPuqh07dn+z/OCGUWe7/CYwZ2Zx6Euol+fPDC2ukiqDZvW4vltTVreFzf2fgYI+TroXqMPB6KODRQFLwL2laa3EpXgS0d91qDdXsViFD7Lbjl6IXAcplzCTAz0JlgDXAOx0/0ByiZTVA9tXAi23xo9vtAt/nIHMwfI5xvA/5giwjr8/UIIzpLvpD9YGEPwAwUK+VZEt7wtlCkVO5ADPvzzL4Mi3pMlMc3QZrEERDJ5jE6AF6tvQp/JCAe08CD4UA8oNWfsddzFUjFH0rD65Oxvb0WGJqN1rIAGW1jAUjVJk6QEP/y/TmRikdL7nlA0A/n/0pElYhgaO7U/FBgOXAPI3s63qfUTQkqp1xAF88YRsEKcP+AwBhGALB5kjDj4vUbzA3Z797bb98eH/UvWLICxxI8SBPZpgKF3FDuFd1KxRq4VJy7x7zVyGGvT96+7Ug2QeoDAXE+g7IubO8lfOgBxhIDmIIRUUXdljrjob9YUNyDAl2CBxSmCajJ3oHP3Wl47gfsDQyj5UmGzwow3MhBZg6LnRTPQrkl4dwryKVnX0EDAWKxC5zrApGI1nsXenXxi/Lh1UnX6A4UCgP3Xjraf/joJa5xFQBd484nqbYrLLVussKGhkNN1TXPxWZ2Jj34KPpr9qfkmyOavQxBwSMD+BIt1lrvDI1VoguGnxILOCNy/p6kJqs/C8Cu76yNjSJHWLoA5DlAmZps3MEzVsWTbxxATGbs1rh2t2MFmQ0CXRjiKImGs5k5Y/uj3hQFEjxpP27EMtJ83qmZXkhnrc+hkNtqpU2Us6OqR5uY2LiFVyffnZzaoIPs05Oz71/9dPxGbCfD4GtnLZTSNu/j8rFtTWtG5Tcubw3KFk6sxwhqZNeUSS13sLaS4/NzWMT7X87q+FnnoJFdaeUjfFB13koHAg/MNHoMfMovP+tzEL+N+nM0WEajvjBYNrObbrKYbY8xf9fauzRwzDqrNbH+KFl0c0IEnpt9OmvL0Mi0vsDPzhSD+QFmrmotdXoII6E/7Y8QV/3pcGL2Z5uxRQbOFQfRmmrWmorvkWmWsci5l1Yd9AvSujHneB4ZdJpCVJZdu+lWU1st3Cx1uLGXgV5qmAYi1RL6VnMkyU1D2b+LxSkPnTsyBhpa60zUPCijFTRUfss56Hewq69AqQT/gHNSqi2GsQjyqDKrhFEB+1DpJErVp/JHjDYElWFkZCJa7BwwUxadyKISBh5SxpwKXAaLB10gqkoOKAVxIKtJyEKhegQyOdE0AEuZJb6KbXcvw2TphBUwXUlGRU6beFxNPpqeJtqYrGbvdfQTuIFEa3HlLAejyaC+67Jcgq78mYqeUmCg6VryXInZwRSMD493L0EgSvON3AhZFOOlwQ16Gyxx3WLlxO49+1yApWcJ23o6nKLLNphOe2Z/rIxrtI+TFbdzXFKGwStZdmGyK8C/7QVR+cBP+Wd76WRgFvjDAbGv8TN3/ww/npvsZ9B3QFHy9DKFL/LqCnAAF4s/oXMmqYN9JFlgASorq+qDmtLfBomqKnhsjOsDSoHa9iVmierRMziGPM2fGc+2qLVaeJGsP11RKgiPAhD/v328XAClSfFJpxXMsyeuQ8u2/mY4tfhsy3rI49D5EV2KIMLYfFRkaFe4IRZr8TvHzUPBjpp/kVXCuQIjl5Ox//qP/wRJQtlvkVktRRaCsVb5nSjJQwFBskGYj5rnKVNW6GGSZ8Fef/jJ2hiICYNlIwAjnsnAy3zq8/ly5vhTfz4aDie8PxiN+HLQm4+XPW82no4mgxl3x5Y17vWXvclg6A3GTn866/H5zB25Xm8+9PuTuTcfjJfz+XIy3hh4kfM2Ai7yOR7ccR+PLfzvD/HQ0omikBqcqS9n4hsQTXwRuXuMj2ie81pav8Tbd1TXdfruNSX4yWXBzCNK/GAZymkAoYfKbTlieRDxLvbmnqbsRH2AdBWKyUh5CjjJi0ccJNRRf8zK2sC6Asb6wDa1JNdDWdL0RoYmNvhPKmsvVBQsHIvfsJ5wLbxTaS3SQhd4DC4wobpM0OFLo4yyqG7C74KMsgCYqpXKqYKmmFFoJODGuvqqLQ1MyIDk7iNqa3uZRd1FE7oDVyKeIxZ52qU4KylsDNsAvjAOhU48mATF5RX7eFEvCwEVffFJaI75HBmw36tZW3rvhnKU4UKsYqEalkqpVVYKJaNFSEkFnjSOIMq3my6VwVJBk1nwkrgVTStDpyp0/d+xEOpFT23BCHQBVJr9iDC1ZjPIxvU8tsJiS+LaRt/T+PpVzbdYCJJgSQUSCAxfZNUgD5wQsberCktqqSJFr0d3rmJQ2lgV+G6oiRMq0mEXaNxdsFxWclAVD4VQkGAykkSGIAgB6gHLBHkGpjccnrrh/eL9mehisWOVkWeOD/xA7KtYt6pyufgkqoFTDKVGmsYQSUFUVHJGOPQrkJbciUy2hNPrA7flXR7DYlHtqGpsgObk7CoJPWFTga+FYeZSOMujIY8FirhN50QWYaA4zEBQAGiHrYIVD+kk3MLimBsmcFJBY2iR1Xt2AWaPWOlFAxpFcVdJmssoLvYnjUmZZmxGdnCTaEUVvzIC1wAD2wZ5gbtX4mFV5FbdaaTCDJm/Jq8bk/KWTGxGzsr4mn1lWZni7lhZEdXCEXKuD6lyvmBa2DF6CKi+MUiJgcmzIopIp3t/d1yUmTL9H5R+hQYMBLdfhOibpQFQkbyWK+5eC5oGWDMdIwbK+hp26awEhYO8AQ363ARJkVHNjpsUKUzskRuEaVtEqkJlWWfTDWI/rLSLBgwLPQRZkRPFCdAUlo7n/S14XjWTfBLHWBYSZS3NZTS5lBVPJJYevcC/D1dg+n8QO2VfJBSzWt2DFu/qlJmYEQU2BtOR2e897YjI6H5McsvYw3K30l1ul7A1bJHYO2IvqUqfcgJUBNdtl8KlYSrkhrGHwxslSBv7PxJwxT+CuB4uQy0pii6tbQJXj3qtR0oDWM1eGRTVGlH1x1zchihWLKSLDGFwrS4kgKD9mAWR94maFgxrVOgAgob0boGda6BIEWNZGTjvIbnaQdYVpWTIuHAOb4IMzcXNBjfJ4obJrZ5Ko3s67k9my6m/HM9Ho1lvORkNZuOlO5nNluOhP+KT5dxzlvOZZY08MLrBvna9vjOdeRPfn/cm/eWw5/swcsnH8/Fw1PPmG43ucuaG2V22kOE97JHlDR8UkwOOvEmdyCaGN/BAL9je3+Djrbgrc31ju87KARf1XrrMxLJgDysWRbKBs4x89DWWfb4y0MyXsV2sDCPGbDNl+WGQOn5AA5pW2rqeyiGC316gfhLZnGKF7MiEZkDRdPzm+9fv4Ah5wQ2xUnhvVYu4dWLQ9KK4akvI9MXZ+fHrk5Z4cgkjjChK8DioN2//dnL8ysbsTkuAlIKjDtadfo0UUp3cLKsbBHxhaX1tJBhAQFAtFMZrrVurhq1aDDbwazvf21ubAJ9UddjWrclwJb/w4PIKzvuPM7t3ljgg+iwLJJzMFIzHyB7743H/9+KS6v4IKu39I2IZ40ZULujbeVB2dbfZHSe2wA8BjWKjtSNHs33coLGnNcs4hCLtt4NpMoRabB3mDzcvHUDxKw5cghVQoeFasS3isSbD74AjhEK/VBSqhrmOdjh+SdLrDB5zq20y14KxHThXw0PsfGeyuxj5Kvk7UW826GOeZzYYmPNKO60KItY6QR5akK9dX1E4AjYsVjbMLtOLpMGWBXgMe+vorCkmGVrBCpdlEeJFO1xmJWfh7PwsnGgjdLKyPoxS8vsYgoE1d8waPJLbqt8LUf/IvEJcAAMrJCnC0qJhP58evy3dV0fe99LMkdJVHvQxyUu+JSzmMlAOOPjDXVyMmrB1JzV4lPtF9aPWJmJKjoC3pKPoaYMzi72ifYIJh3dNMrYGEHEkbP5ql4A6F6QiGsWEPbQ5yN5UVwLxbk4aoEduNbRoBKwa4L6ADcnfQJ0p8Fq5llhih7EBWCjgR5jrP/7wc1OPwmSpE+vBbarkxhgwtoHXEQFgsBDhgcVOoiAXqxaVNTV4QuCbMhKfCAf4CoxbETMh5h7OUDTNRj09ibmJu8UBVPVR6/Wj7jXGZZPbbEcvJOs2j4DgfpQO27hfCZ11Q0uKD0xZfZM22f/2BWlT15YFSkNf1VqhJi+L5nY/CkOiK3rLom8gBpWzCYHYRRDduE91bfVCPFaXJyDs6KIaO6qTAhZu0ZVbEjC/iyiFzW6y3ESipmG6lY+l7eaPR0N/0h/OXWfYGw37s6nnz5aDfm86Xk6nk97Y8WazoTOyrCH3p/PZkLvLOR+N/J7v+e7A8WfzKR+Doef2+v2Z5/qTjbZbNXXDeKuaKKNKvE//63VEr3l8TgcMDwAFA0AAZdI2rowr8qNB3hwL+xi4JMOwJV0uTuIbmfwLHTTBglhAuniPycYLGakGAVbAscEEdeKT2LhMYKLPRYBVuctEuKS6f0/3d8Ff35H5LfKIwOsCUYORk1ha2kyWmJVwRaVYnNyCk447FbdqyXiz3x2/PTlbsI94wfiQzT5hefXO/vZec9FLVIB+vr7ZNddDBD3WpSAKKB4HJLO434LFsrCSf3B4+uPBDwc/UzGPlBi7VKyFY0poWBQDcJaBA+z5+bqLjSY7TT6c6EOAV29T8FVpGAwZwJAffgZ5C4zPqEXYYZTdgv8lvR/FgpgE/S0s5aw2CZOMYRLSItCgK69qCKaewGiXo2DIRA4RpX3YvA/eV2Ms7QloX5tuKsdWWWUpu9b15m4Y0XnWKALDZzBcZLclTdo1L/sk8gKGm1JRukKQDbqqumzd29Dp+F3Vp39IOJ8OEeela9826vVP1ajJ4Y4sgsSoOg8TvFyUMBQ+eJtJOyji4GR4OC5uE6qXOi6rAOAREexCQIM2zBlAV4Aa8fRS3lmKyFNVsaCyfFHOkCeJtWGf76sVTw/pqDT6vNGusc8O8dUEB+zkhqq1nACkJZeae3lPSQISyBY7Q1WNKQC8yhLnt0HGpZWlbuFjbwFNyB/cisCFibsTmXFMloTBkm4OgDmDAa0LjcuFBr+wtNcMfPf9mxP7w9+Oz07EXbc3x/92cqpRc3CocgbnZDx7/AZVj3CqHHZLDpBIKwf0UgA3TbIMX/1AJkyeSb8TCyAwT0NlGuMZppn3+5OhTDerkgjM8OIJWqL/6qQA0agXV0pTpMU9+sNHTJjfcOMVXxaXJnsJPMRNsAmpWrXzqZb2+iDM0DJipZe9hmFXhFexNplUswnsEoZAEsAa1ql2VUYggoV8lDkWfzJ6juLDrBIGzeb5J60q9pWOSlQt+oxgdkdo36EFJ9z4VFia2vQ0EqYADKzNXGupTQr6DWajMmVhVqPRaciKlzvmXhXxddbRJqFCjtZJai21Sd6KV0ncdx3gi6gIKYhdbs5i39Fd2i5aCCsO/2LBw0EZgNbKolV8jcLrr18cZCpSi+pflV8iYvBA+PwW3/1BGEWrO0yS6woU930w6vEogloNUYKrO4zkaAC/5VcRp/dphAkA0intuO1I0BtqOPggLgbL9xDQ8ZShYNgkhva1JKl6o4QoptCIlRboM/BY0ion4rm4WPIH6NUrFA3PmFGmCEwteizLokaUppvOK2n8+EnwUvKiKH0qQYMSkswoD+i9smdS8boVaRrBUw3rd9wVAXdxLYsIKVwTMMvQQtLWa4GxRc6TKB7InHthAmnIKpMctXprUdGVhLyrXEzKFITOClUuBdYJZShzq2wCLwWzxX6KMRCqJZJLaxBjxQTzwhSE5HduWHhSb1D43sV/iV8yIr3lpsQkZjFBhmnTZTI5hQBuk/SaInRozuEBACMfc+j7FXtUUfwyp+tTLr1K6b5KVH2UkMl/FFEBIVhwGvBFZrpFlqHmcq9EHndAkar5YG7OtnqDVBoQY52RyMuies2KZVdpZYPc6Ig7MWABEy5lPq+8b5zEnUUNoLKHyCI0hV2F383SWNa1IlVKGWROHmRBWBzUYKGRc+B4XodKR8CCwDcQYXqT4Iz73cn0n2nRRDBhsTt4wS8Dhx/4DczszKr7rgY6hbl9WZhMfPNi9Y2DYsGa5LpvuX2A+hpGa0PLWllk1sXiVSFS2YvFv5+cvm943f9n/Yg9RiQ59ufjeS1V084geKoPmar+Pz95c/L25Pz03/ACgJNdZ2WQha78W3XflgKFSk4eEdMwSktt8PZL6FtKa9dg1n7u7YnicSFE7fI2t9HBabHXWiSDUk4ojO025Y++A9af4PeNA1VA+mNPqpONPUUR5VN6ohqqd9x/6qrn2qrnmweurXpLz7VVb+lZXzV2ZDVOOlNaD0UqSsf8NhEvBitlwz29CQztAQ7e840TrofqBANPJhjLnU8nZn/0GP9WGXYTXBF8XRhNcIWLQM2NOsViF3XNuVi0Jd/pzQwp2rHSk5d1QaDtuBBtcREt0fSisCEV3qSUjCf9GHI/r4vKgmeouKw6RoXVhufF8oIb2+VBaHw4PQHb/o394vj85d8adfG4l6My6bEqtX8tkV+dDBsHGAO8XYr/qkhSp5HNPgcSVQoWY6Bg+IRFJrYOqo/Ov7AiVpg2RgOAfjZA0RQHtDn5BgxRySO1qYAdgs0voxxIN6mpG7BQyVbxElBZwg07FGYOFWNh1TLOxaR3LdQVqvcGNGQD9LUxXKtCP1rK/rZyIvAlaymwpdWAcex5KjQuPfQ3b7sElLCkR5UIEWjMrtbANCm0z5BAxibPrkY8/CE4Z5/1O53ffG9FVIDRbZh+rz96SiR5/ZJGS+6kIeEojn7EemQdt8gS2T6Q7Y0EILoYaMUxLEg2ZMGy/tF5DsOhsf6eHBXgBShU9pqivkYsl1VXjZfCWCm4iKkno8m9tShu9R458JHttAh59kwegTUsGf8k7KkFv1ulJvsnkeKUP5aJd79YInPBgp6zLy0YfnTZ5BUM52Ozj37BCGg3as9xNQqA3iSX6NTXsiXq9kTl8Mg0FGZ8QIELFs6knOpYO+uLxcJx4MGYHYEIW5dCZfwfIRwx/7Ox8u07kxkx6wLbAhNjGu+wOejaSqOMwopGs1GdHrO9CSfb0EQFGnre01vl6aa+dKnBvos3tMPKKaWPF3NaO+AO+Cpraa3lB7X7PDf2reQ/UF9UjmjdZuqbSmhU6wKpggRtvB6CfA/kzWcG2qqmCG2Z6yz/WwjUOlAnUmuHilAbmwWxNjZvINjm/hXRNvbRCbexU0m81h6tKJcHQluyJdJD2lWzL5tn1POw2rsc8y25463A8E4JaHMqAwZnCiRQkqFKSoqMshuJr6LRoH85FZzEWwGiyEgoxkq1mqe8yKT2dOg+snLeKz91Kzjpw1aOIl4yYD/OareT0Zc8lEnw7dBynFpeNEKRBVDc+26SYswIFkf+tFZ33s7PKtNhf55tYeknsPVTeZGOqm9/zh7vk4Ftzh/r9yTe3si95bVCzAj/zxFw/Uh7Ta5t79p7DNfaoYMPSof+/6OOkNePdOp/C+VKq28bXX9H9bL2MindHNxvkYlrnkorOM1V/UgK6xNW+4iAmHhs6Cu1bjeppco/1eC0MIgsyAMdpzcGsdbWMsPDzrafGDzmILvIF5EJljx1MHht6lH0c5nyqdxIiq02gOGruDK2S3E16D6Z/LO4OkmSdle8lkS0ov+ZUdJGTAt+qDAVR5OeOcAXiY/7A/zyFDO/ucm6vYIxS/S9RNCSdvvHTIZIybWzRK3qfeyWluYtRnhFX7MBDYMA5GQqFGGKAEcJR/GW4y20+HPBC4yRq8grJc0wVdSAJ96LUcZzI8yiU9wwo4pXuq566wTr5e+PGt8Nqw13t2dk+NZ0ky3pjWB70mlp674+SZTRLJYMdttRZgg4G8Zq/J19zHIM/xAc/Zn+MjirR0V9WZuN3XKGhTF/xOL2d8Ypr0wPasMKKP70Ozle/ZbLBGflPSGV89GyALEIJmDQRyn/SLxs3cM0QgOYcuqp/MrC9G56X49LUGWZDOWXof4gwzRXA5q4XS7OnsxtZTwVt3Lw4hO+slPaN2WqAM4oypu1FzLWUVrjiF4LWrbz3f438tz+78hvLbzWYEG5YssNuZMabUyqAgjrcYedbr2vxrltUqypg5rmgyoarsLNtNVGlk1I1dnQnKNQHY6fGjsR6cZKO5nNLiqZuGppK2OULfaBzj3m06pb5btd295QaOONUHAc13BeNWGxC9Xd0h3wDK+NqtvqeI/7ILevb+Cf4+r5pU3AfH8zrMviAFMwB5R92Q5MrEosLwYexDXQh+Me7vw3xa9pDW9kAAA="""
MODEL_URL = "https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF/resolve/9217f5db79a29953eb74d5343926648285ec7e67/qwen2.5-0.5b-instruct-q8_0.gguf?download=true"
MODEL_BYTES = 675710816
MODEL_SHA256 = "ca59ca7f13d0e15a8cfa77bd17e65d24f6844b554a7b6c12e07a5f89ff76844e"
ROOT = Path("/kaggle/working/wave121")
TREE = ROOT / "repo"
RESULTS = ROOT / "results"
TARGET = ROOT / "target"
MODEL = ROOT / "qwen2.5-0.5b-instruct-q8_0.gguf"
FINAL_ZIP = Path("/kaggle/working/glcuda-t4-wave121-in-process-stability-results.zip")

if ROOT.exists():
    shutil.rmtree(ROOT)
RESULTS.mkdir(parents=True)

def run(cmd, *, cwd=None, env=None, timeout=14400, check=True):
    merged = os.environ.copy()
    if env:
        merged.update({k: str(v) for k, v in env.items()})
    p = subprocess.run([str(x) for x in cmd], cwd=cwd, env=merged, text=True,
                       capture_output=True, timeout=timeout)
    print("$", " ".join(str(x) for x in cmd), flush=True)
    if p.stdout:
        print(p.stdout[-12000:], flush=True)
    if p.stderr:
        print(p.stderr[-12000:], flush=True)
    if check and p.returncode:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p

def save(name, p):
    (RESULTS / name).write_text(
        f"RETURN_CODE {p.returncode}\n\nSTDOUT\n{p.stdout}\n\nSTDERR\n{p.stderr}",
        encoding="utf-8",
    )

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

def archive():
    if FINAL_ZIP.exists():
        FINAL_ZIP.unlink()
    with zipfile.ZipFile(FINAL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
        for path in sorted(RESULTS.rglob("*")):
            if path.is_file():
                z.write(path, path.relative_to(RESULTS))
    digest = sha256_file(FINAL_ZIP)
    print("ARCHIVE", FINAL_ZIP, digest, flush=True)
    return digest

def percentile(values, q):
    values = sorted(values)
    x = (len(values) - 1) * q
    lo, hi = math.floor(x), math.ceil(x)
    return values[lo] if lo == hi else values[lo] * (hi - x) + values[hi] * (x - lo)

def bootstrap_ci(values, seed=118, draws=20000):
    rng = random.Random(seed)
    n = len(values)
    medians = [statistics.median(values[rng.randrange(n)] for _ in range(n))
               for _ in range(draws)]
    return [percentile(medians, 0.025), percentile(medians, 0.975)]

phase = "bootstrap"
try:
    embedded = gzip.decompress(base64.b64decode(PATCH_GZIP_B64))
    if hashlib.sha256(embedded).hexdigest() != PATCH_SHA256:
        raise RuntimeError("embedded patch hash mismatch")
    patch_path = RESULTS / "wave121.patch"
    patch_path.write_bytes(embedded)
    (RESULTS / "source.json").write_text(json.dumps({
        "build": BUILD, "base_rev": BASE_REV, "source_rev": SOURCE_REV,
        "patch_sha256": PATCH_SHA256, "patch_bytes": len(embedded),
    }, indent=2), encoding="utf-8")

    gpu = run(["nvidia-smi", "--query-gpu=index,name,compute_cap,memory.total,driver_version",
               "--format=csv,noheader,nounits"], timeout=60)
    save("nvidia-smi.log", gpu)
    fields = [x.strip() for x in gpu.stdout.splitlines()[0].split(",")]
    if len(fields) < 5 or fields[1] != "Tesla T4" or fields[2] != "7.5":
        raise RuntimeError(f"requires Tesla T4 sm_75, got {fields}")

    phase = "reconstruct"
    clone = run(["git", "clone", "--filter=blob:none", REPO_URL, TREE], timeout=1800)
    save("git-clone.log", clone)
    checkout = run(["git", "checkout", "--detach", BASE_REV], cwd=TREE, timeout=600)
    save("git-checkout.log", checkout)
    applied = run(["git", "apply", "--whitespace=error", patch_path], cwd=TREE)
    save("git-apply.log", applied)
    diff = run(["git", "diff", "--check"], cwd=TREE)
    save("git-diff-check.log", diff)

    cargo_candidates = [shutil.which("cargo"), Path.home() / ".cargo/bin/cargo",
                        "/usr/local/cargo/bin/cargo", "/opt/conda/bin/cargo"]
    cargo = next((str(x) for x in cargo_candidates if x and Path(x).is_file()), None)
    cargo_env = {}
    bootstrapped = False
    if cargo is None:
        bootstrapped = True
        rustup_script = ROOT / "rustup-init.sh"
        urllib.request.urlretrieve("https://sh.rustup.rs", rustup_script)
        cargo_home = ROOT / "cargo-home"
        rustup_home = ROOT / "rustup-home"
        cargo_env = {"CARGO_HOME": cargo_home, "RUSTUP_HOME": rustup_home}
        install = run(["bash", rustup_script, "-y", "--profile", "minimal",
                       "--default-toolchain", "stable", "--no-modify-path"],
                      env=cargo_env, timeout=1800)
        save("rustup-install.log", install)
        cargo = str(cargo_home / "bin/cargo")
    if not Path(cargo).is_file():
        raise RuntimeError(f"cargo unavailable after bootstrap: {cargo}")
    (RESULTS / "cargo-discovery.json").write_text(json.dumps({
        "selected": cargo, "bootstrapped": bootstrapped,
        "candidates": [str(x) for x in cargo_candidates if x],
    }, indent=2), encoding="utf-8")
    common = {**cargo_env, "CARGO_TARGET_DIR": TARGET, "CUDA_VISIBLE_DEVICES": "0"}

    phase = "host-tests"
    tests = run([cargo, "test", "-p", "glcuda", "--lib", "--locked"], cwd=TREE, env=common)
    save("cargo-lib-tests.log", tests)
    if "67 passed" not in tests.stdout or "0 failed" not in tests.stdout:
        raise RuntimeError("unexpected host test summary")

    phase = "cuda-parity"
    parity = run([cargo, "test", "--release", "-p", "glcuda", "--test", "parity",
                  "--locked", "--", "--nocapture", "--test-threads=1"],
                 cwd=TREE, env=common, check=False)
    save("cargo-cuda-parity.log", parity)
    if parity.returncode or "0 failed" not in parity.stdout:
        raise RuntimeError("CUDA parity failed")

    phase = "compiler-resource"
    ptxas = shutil.which("ptxas") or "/usr/local/cuda/bin/ptxas"
    resource = run([ptxas, "-v", "-arch=sm_75", TREE / "glcuda/src/kernels/glcuda_sm75.ptx",
                    "-o", ROOT / "wave121.cubin"], check=False)
    save("ptxas-sm75.log", resource)
    if resource.returncode or "spill stores" not in resource.stderr:
        raise RuntimeError("ptxas resource gate failed")

    phase = "model"
    urllib.request.urlretrieve(MODEL_URL, MODEL)
    model_meta = {"bytes": MODEL.stat().st_size, "sha256": sha256_file(MODEL)}
    if model_meta != {"bytes": MODEL_BYTES, "sha256": MODEL_SHA256}:
        raise RuntimeError(f"model identity mismatch: {model_meta}")
    (RESULTS / "model.json").write_text(json.dumps(model_meta, indent=2), encoding="utf-8")

    phase = "build"
    build = run([cargo, "build", "--release", "-p", "glcuda", "--example",
                 "wave118_in_process_stability", "--locked"], cwd=TREE, env=common)
    save("cargo-build.log", build)
    exe = TARGET / "release/examples/wave118_in_process_stability"
    prod_env = {**common, "GLCUDA_FORCE_Q8": "1", "GLCUDA_GRID2D": "1",
                "GLCUDA_FUSE_Q8_GLUE": "1", "GLCUDA_NTILE128": "1",
                "GLCUDA_BSTAGE": "1", "GLCUDA_GEMM_N16": "1",
                "GLCUDA_GEMM_N16_PREFETCH": "1", "GLCUDA_ATTN_MMA4": "1",
                "GLCUDA_ATTN_MMA4_REGQ": "1", "GLCUDA_ATTN_MMA4_AV": "1"}

    phase = "build"
    build = run([cargo, "build", "--release", "-p", "glcuda", "--example",
                 "wave118_in_process_stability", "--locked"], cwd=TREE, env=common)
    save("cargo-build.log", build)
    exe = TARGET / "release/examples/wave118_in_process_stability"
    prod_env = {**common, "GLCUDA_FORCE_Q8": "1", "GLCUDA_GRID2D": "1",
                "GLCUDA_FUSE_Q8_GLUE": "1", "GLCUDA_NTILE128": "1",
                "GLCUDA_BSTAGE": "1", "GLCUDA_GEMM_N16": "1",
                "GLCUDA_GEMM_N16_PREFETCH": "1", "GLCUDA_ATTN_MMA4": "1",
                "GLCUDA_ATTN_MMA4_REGQ": "1", "GLCUDA_ATTN_MMA4_AV": "1",
                "GLCUDA_TELEMETRY": "1"}

    phase = "lmhead-ab"
    records = []
    orders = [("retained", "candidate"), ("candidate", "retained")] * 3
    for repeat, order in enumerate(orders):
        for arm in order:
            env = dict(prod_env)
            if arm == "candidate":
                env["GLCUDA_LMHEAD_GEMM"] = "1"
            measured = run([exe, MODEL, "profile"], cwd=TREE, env=env, check=False)
            save(f"profile-{repeat}-{arm}.log", measured)
            if measured.returncode or "[wave120-profile]" not in measured.stdout:
                raise RuntimeError(f"{arm} profile failed at repeat {repeat}")
            profile = json.loads(re.search(r"\[wave120-profile\]\s*(\{[^\n]+\})", measured.stdout).group(1))
            stages = [json.loads(x) for x in re.findall(r"\[wave120-stage\]\s*(\{[^\n]+\})", measured.stdout)]
            lm = next(x for x in stages if x["name"] == "lm_head")
            if len(stages) != 9 or profile["oracle_token"] != 3323:
                raise RuntimeError(f"{arm} contract failed: {profile}, {len(stages)} stages")
            records.append({"repeat": repeat, "arm": arm, **profile,
                            "lm_head_ms": lm["total_ms"]})
    by_arm = {arm: [x for x in records if x["arm"] == arm]
              for arm in ("retained", "candidate")}
    med = lambda arm, key: statistics.median(x[key] for x in by_arm[arm])
    summary = {"wave": 121, "gpu": fields, "model": model_meta,
               "records": records,
               "median_gpu_prefill_ms": {arm: med(arm, "gpu_prefill_ms") for arm in by_arm},
               "median_prefill_tps": {arm: med(arm, "gpu_prefill_tps") for arm in by_arm},
               "median_lm_head_ms": {arm: med(arm, "lm_head_ms") for arm in by_arm}}
    summary["speedup"] = (summary["median_gpu_prefill_ms"]["retained"] /
                          summary["median_gpu_prefill_ms"]["candidate"])
    candidate_lm = summary["median_lm_head_ms"]["candidate"]
    if candidate_lm <= 0:
        raise RuntimeError(f"LM-head event was not recorded: {summary['median_lm_head_ms']}")
    summary["lm_head_speedup"] = summary["median_lm_head_ms"]["retained"] / candidate_lm
    summary["target_15000_tps_achieved"] = summary["median_prefill_tps"]["candidate"] >= 15000
    (RESULTS / "wave121-summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print("WAVE121_RESULT", json.dumps(summary, indent=2), flush=True)
    archive()
except Exception:
    (RESULTS / "FAILED.json").write_text(json.dumps({
        "phase": phase, "traceback": traceback.format_exc()}, indent=2), encoding="utf-8")
    archive()
    raise
